# ML-05 — Feature Leakage Check

Audit features to ensure target information does not leak into the inputs.

## 1. Feature-label correlation analysis

We calculate correlations of all numeric features with `is_declining_label`. Any correlation near 1.0 or -1.0 suggests feature leakage.

In [1]:
import pandas as pd, numpy as np
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlations = df[numeric_cols].corrwith(df['is_declining_label']).sort_values(key=abs, ascending=False)
print(correlations.head(10))

is_declining_label        1.000000
days_with_impressions     0.190055
content_age_days         -0.163882
age_tier_order           -0.156142
trend_pct                -0.141068
impressions_last_30d     -0.093980
word_count                0.090157
days_since_last_update    0.081383
char_count                0.072188
clicks_last_30d          -0.071935
dtype: float64


## 2. Verification of exclusions

Confirm that `trend_pct` and `trend_direction` are not in the training features list.

In [2]:
from scripts.ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES
leakage_features = ['trend_pct', 'trend_direction']
for f in leakage_features:
    assert f not in MODEL_NUMERIC_FEATURES, f"Leakage detected: {f} in numeric features"
    assert f not in MODEL_CATEGORICAL_FEATURES, f"Leakage detected: {f} in categorical features"
print("Verification successful. No direct leakage columns are present in model features.")

Verification successful. No direct leakage columns are present in model features.
